In [ ]:
# Core imports
import numpy as np
import torch
import sys
from pathlib import Path
# Add project root to Python path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Import modules
from src.data.dataset import create_data_loaders
from src.models.loggptmodel import LogGPTModel
from src.engine.trainer import LogSeqTrainer
from src.utils.metrics import evaluate_model, print_metrics, save_experiment_results
from src.utils.data_loader import create_train_val_test_split, filter_normal_samples, load_loghub
from src.utils.visualizer import UniversalAnomalyVisualizer

In [ ]:
# Define paths
DATA_DIR = '../data/hdfs/preprocessed'

In [ ]:
# Load HDFS data using the utility function 
X, y, vocab = load_loghub(DATA_DIR)

vocab_size = len(vocab)
print(f"Vocab size: {vocab_size}")
print(f"Events: {sorted(vocab.keys())[:10]}")

In [ ]:
# Split data (70/15/15)
splits = create_train_val_test_split(X, y, train_ratio=0.7, val_ratio=0.15, random_state=42)
(X_train, y_train), (X_val, y_val), (X_test, y_test) = splits['train'], splits['val'], splits['test']

# Filter to keep only normal samples for semi-supervised training
X_train, y_train = filter_normal_samples(X_train, y_train, verbose=True)

In [ ]:
# Convert strings to integers using the loaded vocab
def convert_to_ids(sequences, vocab):
    return [[vocab[event] for event in seq] for seq in sequences]

X_train_ids = convert_to_ids(X_train, vocab)
X_val_ids = convert_to_ids(X_val, vocab)
X_test_ids = convert_to_ids(X_test, vocab)

In [ ]:
# Pad sequences (integers)
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_len = int(np.percentile([len(s) for s in X_train_ids], 95))

X_train_padded = pad_sequences(X_train_ids, maxlen=max_len, padding='post', value=0)
X_val_padded = pad_sequences(X_val_ids, maxlen=max_len, padding='post', value=0)
X_test_padded = pad_sequences(X_test_ids, maxlen=max_len, padding='post', value=0)

print(f"Max length: {max_len}")
print(f"Train shape: {X_train_padded.shape}")

In [ ]:
# Create data loaders
batch_size = 64
train_loader, val_loader, test_loader = create_data_loaders(
    X_train_padded, y_train,
    X_val_padded, y_val,
    X_test_padded, y_test,
    batch_size=batch_size
)

In [ ]:
#  Load the pre-computed embeddings
semantic_vectors = torch.load(f'{DATA_DIR}/semantic_embeddings.pt')
emb_dim = semantic_vectors.shape[1] 
print(f"Loaded semantic embeddings with dim: {emb_dim}")

In [ ]:
model = LogGPTModel(
        vocab_size=vocab_size,
        embedding_dim=emb_dim,
        num_layers=6,
        num_heads=8,
        dropout=0.1,
        semantic_embeddings=semantic_vectors
    )

In [ ]:
# Print model summary
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nModel Architecture:")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Embedding dimension: {emb_dim}")
print(f"  Semantic embeddings: {'✓ Loaded' if semantic_vectors is not None else '✗ Random init'}")

In [ ]:
# Train
device = 'mps' if torch.backends.mps.is_available() else 'cpu'
learning_rate = 0.001
patience = 20

print(f"Training on device: {device}")

trainer = LogSeqTrainer(model, device=device, learning_rate=learning_rate)

history = trainer.fit(
    train_loader, val_loader,
    num_epochs=50,
    early_stopping_patience=patience,
    print_every=5  # Print every 5 epochs
)

In [ ]:
# Evaluate
# Capture predictions, true labels, AND anomaly scores from the loader
top_k = 5
predictions, true_labels, anomaly_scores = trainer.detect_anomalies(
    test_loader,
    top_k=top_k,
    return_scores=True
)

metrics = evaluate_model(predictions, true_labels) 
print_metrics(metrics)


In [ ]:
# Save experiment results
save_experiment_results(
    filepath="../results/hdfs_loggpt_results.json",
    dataset="HDFS",
    model_name="LogGPT",
    device=device,
    model=model,
    history=history,
    y_train=y_train,
    y_val=y_val,
    y_test=y_test,
    max_len=max_len,
    metrics=metrics,
    batch_size=batch_size,
    learning_rate=learning_rate,
    patience=patience,
    top_k=top_k,
    # LogAnomaly-specific params
    use_attention=True,
    n_heads=4
)

In [ ]:
# Save model checkpoint
save_dir = Path('../mdls')
save_dir.mkdir(parents=True, exist_ok=True)
save_path = save_dir / 'loggpt_checkpoint.pt'
trainer.save_model(str(save_path))